# js.scrollbar

> JavaScript generator for custom scrollbar interaction (drag thumb, click track).

In [ ]:
#| default_exp js.scrollbar

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionUrls

In [ ]:
#| export
def generate_scrollbar_js(
    ids: VirtualCollectionHtmlIds,   # HTML IDs for this collection
    urls: VirtualCollectionUrls,     # URL bundle (for nav_to_index)
) -> str:  # JavaScript code fragment
    """Generate JS for custom scrollbar: thumb positioning from hidden input + drag/click interaction."""
    return f"""
        // === Custom Scrollbar ===
        (function() {{
            let _isDragging = false;
            let _lastNavIndex = -1;

            function _getTrack() {{ return document.getElementById('{ids.scrollbar_track}'); }}
            function _getThumb() {{ return document.getElementById('{ids.scrollbar_thumb}'); }}
            function _getWindowStartInput() {{ return document.getElementById('{ids.window_start_input}'); }}

            function _positionThumbFromState() {{
                // Read state from DOM and position thumb via direct style manipulation.
                // Called after each HTMX settle — the hidden input carries the latest window_start.
                const track = _getTrack();
                const thumb = _getThumb();
                const input = _getWindowStartInput();
                if (!track || !thumb || !input) return;

                const totalItems = parseInt(track.dataset.totalItems || '0');
                const visibleRows = parseInt(track.dataset.visibleRows || '1');
                const windowStart = parseInt(input.value || '0');

                if (totalItems <= 0 || visibleRows >= totalItems) {{
                    thumb.style.top = '0%';
                    thumb.style.height = '100%';
                    return;
                }}

                const thumbHeightPct = Math.max(
                    (visibleRows / totalItems) * 100,
                    (track.offsetHeight > 0) ? (24 / track.offsetHeight) * 100 : 4
                );
                const maxStart = Math.max(1, totalItems - visibleRows);
                const thumbTopPct = (windowStart / maxStart) * (100 - thumbHeightPct);

                thumb.style.top = thumbTopPct.toFixed(2) + '%';
                thumb.style.height = thumbHeightPct.toFixed(2) + '%';
            }}

            function _calcTargetIndex(pointerY) {{
                const track = _getTrack();
                if (!track) return 0;
                const rect = track.getBoundingClientRect();
                const totalItems = parseInt(track.dataset.totalItems || '0');
                const visibleRows = parseInt(track.dataset.visibleRows || '1');
                const maxStart = Math.max(0, totalItems - visibleRows);
                const relY = Math.max(0, Math.min(pointerY - rect.top, rect.height));
                return Math.round((relY / rect.height) * maxStart);
            }}

            function _updateThumbPosition(pointerY) {{
                // Direct DOM update for real-time visual feedback during drag
                const track = _getTrack();
                const thumb = _getThumb();
                if (!track || !thumb) return;
                const rect = track.getBoundingClientRect();
                const thumbHeight = thumb.offsetHeight;
                const maxTop = rect.height - thumbHeight;
                const relY = Math.max(0, Math.min(pointerY - rect.top - thumbHeight / 2, maxTop));
                thumb.style.top = relY + 'px';
            }}

            function _navToIndex(index) {{
                if (index === _lastNavIndex) return;  // Deduplicate
                _lastNavIndex = index;
                htmx.ajax('POST', '{urls.nav_to_index}', {{
                    swap: 'none',
                    values: {{ target_index: index }}
                }});
            }}

            // --- Document-level drag handlers ---

            function _onDragMove(evt) {{
                if (!_isDragging) return;
                evt.preventDefault();
                _updateThumbPosition(evt.clientY);
                const index = _calcTargetIndex(evt.clientY);
                _navToIndex(index);
            }}

            function _onDragEnd(evt) {{
                if (!_isDragging) return;
                _isDragging = false;
                _lastNavIndex = -1;
                document.removeEventListener('pointermove', _onDragMove);
                document.removeEventListener('pointerup', _onDragEnd);
                document.removeEventListener('pointercancel', _onDragEnd);
                const thumb = _getThumb();
                if (thumb) thumb.style.cursor = '';
                _setupScrollbar();
            }}

            function _setupScrollbar() {{
                const track = _getTrack();
                const thumb = _getThumb();
                if (!track || !thumb) return;

                if (track._scrollbarAbort) track._scrollbarAbort.abort();
                const controller = new AbortController();
                track._scrollbarAbort = controller;
                const opts = {{ signal: controller.signal }};

                // --- Thumb drag start ---
                thumb.addEventListener('pointerdown', function(evt) {{
                    if (evt.button !== 0) return;
                    evt.preventDefault();
                    evt.stopPropagation();
                    _isDragging = true;
                    _lastNavIndex = -1;
                    thumb.style.cursor = 'grabbing';
                    document.addEventListener('pointermove', _onDragMove);
                    document.addEventListener('pointerup', _onDragEnd);
                    document.addEventListener('pointercancel', _onDragEnd);
                }}, opts);

                // --- Track click (jump to position) ---
                track.addEventListener('pointerdown', function(evt) {{
                    if (evt.button !== 0) return;
                    const currentThumb = _getThumb();
                    if (currentThumb && (evt.target === currentThumb || currentThumb.contains(evt.target))) return;
                    evt.preventDefault();
                    const index = _calcTargetIndex(evt.clientY);
                    _lastNavIndex = -1;
                    _navToIndex(index);
                }}, opts);
            }}

            // Position thumb on initial load
            _positionThumbFromState();
            _setupScrollbar();

            // After each HTMX settle, re-position thumb from updated hidden input
            document.body.addEventListener('htmx:afterSettle', function() {{
                if (!_isDragging) {{
                    _positionThumbFromState();
                    _setupScrollbar();
                }}
            }});
        }})();
    """

## Tests

In [ ]:
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionUrls

ids = VirtualCollectionHtmlIds(prefix="t")
urls = VirtualCollectionUrls(nav_to_index="/vc/nav_to_index")

js = generate_scrollbar_js(ids, urls)
assert 't-scrollbar-track' in js
assert 't-scrollbar-thumb' in js
assert 't-window-start-input' in js
assert '/vc/nav_to_index' in js
assert '_positionThumbFromState' in js
assert '_onDragMove' in js
assert '_onDragEnd' in js
assert 'htmx:afterSettle' in js
assert 'htmx.ajax' in js
assert '_isDragging' in js
print("Scrollbar JS generation test passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()